# Example 11.1 / 4: Recognize Common Quadratic Patterns

Companion notebook for **11.1 Quadratic Models** in the AMPL lab Documentation, section "Recognize Common Quadratic Patterns". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Recognize Common Quadratic Patterns

A mean-variance pattern with covariance $\Sigma=\begin{pmatrix}0.04&0.01\\0.01&0.09\end{pmatrix}$
(symmetric, eigenvalues positive) and a fully invested budget gives the minimum-variance weights
$(0.08/0.11,\ 0.03/0.11)\approx(0.727273,\ 0.272727)$:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set ASSETS;                  # investable assets
param cov {ASSETS, ASSETS};  # covariance of returns; symmetric, PSD

var Weight {ASSETS} >= 0;  # fraction invested per asset, no short sales

minimize Variance:  # portfolio variance
    sum {i in ASSETS, j in ASSETS} cov[i,j] * Weight[i] * Weight[j];

subject to FullyInvested:  # fractions sum to 1
    sum {i in ASSETS} Weight[i] = 1;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["ASSETS"] = ["stock", "bond"]
ampl.param["cov"] = {("stock", "stock"): 0.04, ("stock", "bond"): 0.01,
                     ("bond", "stock"): 0.01, ("bond", "bond"): 0.09}

In [5]:
# run.py
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result)
print({k: round(v, 6) for k, v in ampl.var["Weight"].to_dict().items()})
print(round(ampl.obj["Variance"].value(), 6))

solved
{'bond': 0.272727, 'stock': 0.727273}
0.031818
